In [1]:
!pip install snowflake-connector-python snowflake-sqlalchemy sqlalchemy

In [2]:
import snowflake.connector

In [3]:
import requests
import pandas as pd

In [4]:
from datetime import datetime

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [6]:
import re

In [7]:
from snowflake.connector.pandas_tools import write_pandas

Extract Layer

In [8]:
roles = [
    "data analyst",
    "data scientist",
    "data engineer",
    "business analyst",
    "machine learning engineer",
    "analytics engineer",
    "data specialist",
    "research analyst"
    "business intelligence analyst",
    "data architect",
    "data warehouse engineer"
]

In [9]:

url = "https://data.usajobs.gov/api/search"

headers = {
    "User-Agent": "mhaftran@gmail.com",
    "Authorization-Key": "NK21ZnEiSAktOfnXLZwzGYsW2mf/U5ouL3hkCort544="
}

all_jobs = []

for role in roles:


    params = {
            "Keyword": role,
            "ResultsPerPage": 2800,
            "Page": 1,
        }

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    jobs = data["SearchResult"]["SearchResultItems"]

    for job in jobs:
        obj = job["MatchedObjectDescriptor"]

        all_jobs.append({
            "role": role,


            "title": obj.get("PositionTitle"),
            "location": obj.get("PositionLocationDisplay"),
            "description": obj["UserArea"]["Details"].get("JobSummary"),


            "posted_date": obj.get("PublicationStartDate"),
            "close_date": obj.get("ApplicationCloseDate"),


            "job_grade": obj.get("JobGrade"),


            "salary_min": obj.get("PositionRemuneration", [{}])[0].get("MinimumRange"),
            "salary_max": obj.get("PositionRemuneration", [{}])[0].get("MaximumRange"),
            "salary_interval": obj.get("PositionRemuneration", [{}])[0].get("RateIntervalCode"),


            "employment_type": obj.get("PositionSchedule", [{}])[0] if isinstance(obj.get("PositionSchedule"), list) else obj.get("PositionSchedule"),

        })

In [10]:
df = pd.DataFrame(all_jobs)
print(df.shape)
df.head()

(1093, 11)


,role,title,location,description,posted_date,close_date,job_grade,salary_min,salary_max,salary_interval,employment_type
0,data analyst,"Data Analyst, GS-0301- 13 FPL 13 (MP)",Multiple Locations,This position is located in Federal Student Ai...,2026-04-01T09:15:36.5770,2026-04-07T23:59:59.9970,[{'Code': 'GS'}],108173,158322,PA,"{'Name': '', 'Code': '1'}"
1,data analyst,"Data Analyst, GS-0301- 13 FPL 13 (DE)",Multiple Locations,This position is located in Federal Student Ai...,2026-04-01T09:16:01.7970,2026-04-07T23:59:59.9970,[{'Code': 'GS'}],108173,158322,PA,"{'Name': '', 'Code': '1'}"
2,data analyst,Financial Data Analyst,"Washington, District of Columbia",This position is located at Department of Hous...,2026-03-23T00:00:00.0000,2026-04-06T23:59:59.9970,[{'Code': 'GS'}],143913,187093,PA,"{'Name': '', 'Code': '1'}"
3,data analyst,Financial Data Analyst,"Washington, District of Columbia",This position is located in the Department of ...,2026-03-23T00:00:00.0000,2026-04-06T23:59:59.9970,[{'Code': 'GS'}],143913,187093,PA,"{'Name': '', 'Code': '1'}"
4,data analyst,Program Analyst (Data Analytics),"Washington, District of Columbia",This position is located in the Department of ...,2026-04-01T00:00:00.0000,2026-04-07T23:59:59.9970,[{'Code': 'GS'}],143913,187093,PA,"{'Name': '', 'Code': '1'}"


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1093 entries, 0 to 1092
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   role             1093 non-null   object
 1   title            1093 non-null   object
 2   location         1093 non-null   object
 3   description      1093 non-null   object
 4   posted_date      1093 non-null   object
 5   close_date       1093 non-null   object
 6   job_grade        1093 non-null   object
 7   salary_min       1093 non-null   object
 8   salary_max       1093 non-null   object
 9   salary_interval  1093 non-null   object
 10  employment_type  1093 non-null   object
dtypes: object(11)
memory usage: 94.1+ KB


**Transform Layer**

Transform Job Grade

In [12]:
df["job_grade"] = df["job_grade"].apply(
    lambda x: x[0]["Code"] if isinstance(x, list) and len(x) > 0 else x
)

Transform Employment Type

In [13]:
df["employment_type"] = df["employment_type"].apply(
    lambda x: x.get("Code") if isinstance(x, dict) else x
)

Change Date Format

In [14]:
df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce").dt.date
df["close_date"] = pd.to_datetime(df["close_date"], errors="coerce").dt.date

In [15]:
df

,role,title,location,description,posted_date,close_date,job_grade,salary_min,salary_max,salary_interval,employment_type
0,data analyst,"Data Analyst, GS-0301- 13 FPL 13 (MP)",Multiple Locations,This position is located in Federal Student Ai...,2026-04-01,2026-04-07,GS,108173,158322,PA,1
1,data analyst,"Data Analyst, GS-0301- 13 FPL 13 (DE)",Multiple Locations,This position is located in Federal Student Ai...,2026-04-01,2026-04-07,GS,108173,158322,PA,1
2,data analyst,Financial Data Analyst,"Washington, District of Columbia",This position is located at Department of Hous...,2026-03-23,2026-04-06,GS,143913,187093,PA,1
3,data analyst,Financial Data Analyst,"Washington, District of Columbia",This position is located in the Department of ...,2026-03-23,2026-04-06,GS,143913,187093,PA,1
4,data analyst,Program Analyst (Data Analytics),"Washington, District of Columbia",This position is located in the Department of ...,2026-04-01,2026-04-07,GS,143913,187093,PA,1
...,...,...,...,...,...,...,...,...,...,...,...
1088,data architect,"Chief, Cyber Strategy and Policy, Mission Delta 6","Schriever AFB, Colorado",Do not email applications. To submit your resu...,2026-03-31,2026-04-07,NH,90961,140616,PA,1
1089,data architect,CIVIL ENGINEER,Multiple Locations,"Click on ""Learn more about this agency"" button...",2025-12-22,2026-12-21,GS,74678,192331,PA,1
1090,data architect,Healthcare Engineer,"Arlington, Texas",This position serves as the Deputy Capital Ass...,2026-04-02,2026-04-13,GS,156151,203003,PA,1
1091,data warehouse engineer,INTERDISCIPLINARY ENGINEER/PHYSICIST/SCIENTIST,"China Lake, California",You will serve as a the Subject Matter Expert ...,2026-03-31,2026-04-06,DP,146632,197200,PA,1


Drop Duplicates

In [16]:
df = df.drop_duplicates()

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1068 entries, 0 to 1092
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   role             1068 non-null   object
 1   title            1068 non-null   object
 2   location         1068 non-null   object
 3   description      1068 non-null   object
 4   posted_date      1068 non-null   object
 5   close_date       1068 non-null   object
 6   job_grade        1068 non-null   object
 7   salary_min       1068 non-null   object
 8   salary_max       1068 non-null   object
 9   salary_interval  1068 non-null   object
 10  employment_type  1068 non-null   object
dtypes: object(11)
memory usage: 100.1+ KB


Extract key skills

In [18]:
df_1=df.copy()

In [19]:
skill_filter = [
    "python", "sql", "tableau", "power bi", "excel",
    "aws", "spark", "snowflake", "machine learning"
]

In [20]:
texts = df_1["description"].fillna("").str.lower()

In [21]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
X = vectorizer.fit_transform(df_1["description"].fillna("").str.lower())

In [22]:
feature_names = vectorizer.get_feature_names_out()

In [23]:
# Get top keywords. I try to give those skill filter a boost
def get_top_keywords(row, top_n=10):
    row = row.toarray().flatten()

    scored_words = []

    for i, score in enumerate(row):
        word = feature_names[i]

        if score > 0:
            # base TF-IDF score
            final_score = score

            # BOOST if it's a skill
            if word in skill_filter:
                final_score *= 3

            scored_words.append((word, final_score))

    # sort by final score
    scored_words.sort(key=lambda x: x[1], reverse=True)

    return [word for word, score in scored_words[:top_n]]

In [24]:
df_1["top_keywords"] = [
    get_top_keywords(X[i]) for i in range(X.shape[0])
]

In [25]:
df_1

,role,title,location,description,posted_date,close_date,job_grade,salary_min,salary_max,salary_interval,employment_type,top_keywords
0,data analyst,"Data Analyst, GS-0301- 13 FPL 13 (MP)",Multiple Locations,This position is located in Federal Student Ai...,2026-04-01,2026-04-07,GS,108173,158322,PA,1,"[customer, insights, fsa, analyst, ombudsman, ..."
1,data analyst,"Data Analyst, GS-0301- 13 FPL 13 (DE)",Multiple Locations,This position is located in Federal Student Ai...,2026-04-01,2026-04-07,GS,108173,158322,PA,1,"[customer, insights, fsa, analyst, ombudsman, ..."
2,data analyst,Financial Data Analyst,"Washington, District of Columbia",This position is located at Department of Hous...,2026-03-23,2026-04-06,GS,143913,187093,PA,1,"[open, announcement, hud, housing, urban, deta..."
3,data analyst,Financial Data Analyst,"Washington, District of Columbia",This position is located in the Department of ...,2026-03-23,2026-04-06,GS,143913,187093,PA,1,"[apply, open, announcement, hud, housing, urba..."
4,data analyst,Program Analyst (Data Analytics),"Washington, District of Columbia",This position is located in the Department of ...,2026-04-01,2026-04-07,GS,143913,187093,PA,1,"[non, security, dhs, bu, trade, strategy, econ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1088,data architect,"Chief, Cyber Strategy and Policy, Mission Delta 6","Schriever AFB, Colorado",Do not email applications. To submit your resu...,2026-03-31,2026-04-07,NH,90961,140616,PA,1,"[ussf, force, hire, direct, linkedin, website,..."
1089,data architect,CIVIL ENGINEER,Multiple Locations,"Click on ""Learn more about this agency"" button...",2025-12-22,2026-12-21,GS,74678,192331,PA,1,"[button, learn, click, important, additional, ..."
1090,data architect,Healthcare Engineer,"Arlington, Texas",This position serves as the Deputy Capital Ass...,2026-04-02,2026-04-13,GS,156151,203003,PA,1,"[asset, capital, healthcare, manager, industry..."
1091,data warehouse engineer,INTERDISCIPLINARY ENGINEER/PHYSICIST/SCIENTIST,"China Lake, California",You will serve as a the Subject Matter Expert ...,2026-03-31,2026-04-06,DP,146632,197200,PA,1,"[ops, lake, matter, subject, group, expert, ch..."


=> This didn't work, as the keywords extracted were not important skills

Mapping method

In [26]:
skills = [
    # programming
    "python", "r", "sql", "java",

    # data tools
    "excel", "tableau", "power bi",

    # cloud / engineering
    "aws", "azure", "snowflake", "spark", "databricks",

    # analytics / ml
    "data analysis", "statistics", "machine learning",
    "regression", "forecasting",

    # business
    "data visualization", "reporting", "etl"
]


In [27]:
'''
df["skills_found"] = df["description"].str.lower().apply(
    lambda text: [
        skill for skill in skills
        if re.search(rf"\b{re.escape(skill)}\b", text) # make the keyword matching more precise
    ]
)
'''

'\ndf["skills_found"] = df["description"].str.lower().apply(\n    lambda text: [\n        skill for skill in skills\n        if re.search(rf"\x08{re.escape(skill)}\x08", text) # make the keyword matching more precise\n    ]\n)\n'

In [28]:
'''
df["skills_found"] = df["skills_found"].apply(
    lambda x: x if isinstance(x, list) else []
)
'''

'\ndf["skills_found"] = df["skills_found"].apply(\n    lambda x: x if isinstance(x, list) else []\n)\n'

**Load**

In [29]:
df.to_csv("job_skills.csv", index=False)

In [30]:


conn = snowflake.connector.connect(
        user="Haftran",
        password="JDE64qpVuMYSqW2",
        account="BXAIJSU-ZVC61716"
)

cur = conn.cursor()

In [31]:
cur.execute("USE DATABASE JOB_DB")
cur.execute("USE SCHEMA PUBLIC")

In [32]:
df.columns = df.columns.str.upper().str.strip()
df = df.reset_index(drop=True)
df = df.rename(columns={"ROLE": "JOB_ROLE"})


In [33]:
df

,JOB_ROLE,TITLE,LOCATION,DESCRIPTION,POSTED_DATE,CLOSE_DATE,JOB_GRADE,SALARY_MIN,SALARY_MAX,SALARY_INTERVAL,EMPLOYMENT_TYPE
0,data analyst,"Data Analyst, GS-0301- 13 FPL 13 (MP)",Multiple Locations,This position is located in Federal Student Ai...,2026-04-01,2026-04-07,GS,108173,158322,PA,1
1,data analyst,"Data Analyst, GS-0301- 13 FPL 13 (DE)",Multiple Locations,This position is located in Federal Student Ai...,2026-04-01,2026-04-07,GS,108173,158322,PA,1
2,data analyst,Financial Data Analyst,"Washington, District of Columbia",This position is located at Department of Hous...,2026-03-23,2026-04-06,GS,143913,187093,PA,1
3,data analyst,Financial Data Analyst,"Washington, District of Columbia",This position is located in the Department of ...,2026-03-23,2026-04-06,GS,143913,187093,PA,1
4,data analyst,Program Analyst (Data Analytics),"Washington, District of Columbia",This position is located in the Department of ...,2026-04-01,2026-04-07,GS,143913,187093,PA,1
...,...,...,...,...,...,...,...,...,...,...,...
1063,data architect,"Chief, Cyber Strategy and Policy, Mission Delta 6","Schriever AFB, Colorado",Do not email applications. To submit your resu...,2026-03-31,2026-04-07,NH,90961,140616,PA,1
1064,data architect,CIVIL ENGINEER,Multiple Locations,"Click on ""Learn more about this agency"" button...",2025-12-22,2026-12-21,GS,74678,192331,PA,1
1065,data architect,Healthcare Engineer,"Arlington, Texas",This position serves as the Deputy Capital Ass...,2026-04-02,2026-04-13,GS,156151,203003,PA,1
1066,data warehouse engineer,INTERDISCIPLINARY ENGINEER/PHYSICIST/SCIENTIST,"China Lake, California",You will serve as a the Subject Matter Expert ...,2026-03-31,2026-04-06,DP,146632,197200,PA,1


In [34]:


write_pandas(
    conn,
    df,
    "JOB_SKILLS",
    auto_create_table=True
)

(True,
 1,
 1068,
 [('snowpark_temp_stage_w1sl90eq0j/file0.txt',
   'LOADED',
   1068,
   1068,
   1,
   0,
   None,
   None,
   None,
   None)])